# ViMedAQA baseline RAG (VinaLLaMA + BGE-M3)
Settings: **Accelerator = GPU T4**, **Internet = On**. Run cells top to bottom.
If you saved the index as a Kaggle Dataset named `vimedrag-index`, add it via *Add Input* and skip the build cell.
BGE-M3 có thể *Add Input* (Models/Datasets) để khỏi tải lại; cell 6 tự phát hiện, không thì tải `BAAI/bge-m3`.

In [ ]:
!nvidia-smi

In [ ]:
REPO_URL = "https://github.com/LeMinhNhat2901/VinmedRAG.git" 
import os
WORK = "/kaggle/working/vimedrag"
if os.path.isdir(WORK):
    !git -C {WORK} pull
else:
    !git clone {REPO_URL} {WORK}
os.chdir(WORK)
!git log --oneline -1

In [ ]:
!pip install -q -U bitsandbytes bert-score accelerate

In [ ]:
import torch, transformers, bitsandbytes; print(torch.__version__, transformers.__version__, bitsandbytes.__version__)

In [ ]:
!python -m pytest -q

In [ ]:
import glob

# Index đã lưu thành Kaggle Dataset (thư mục có cả meta.json lẫn embeddings.npy), nếu không thì build vào /kaggle/working
_idx = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/meta.json", recursive=True)
              if os.path.exists(os.path.join(os.path.dirname(p), "embeddings.npy")))
INDEX_DIR = _idx[0] if _idx else "/kaggle/working/index"
RUNS_DIR = "/kaggle/working/runs"

# BGE-M3: dùng bản đã Add Input nếu có (thư mục chứa config.json + tokenizer), không thì tải từ HuggingFace
_emb = sorted(os.path.dirname(p) for p in glob.glob("/kaggle/input/**/config.json", recursive=True)
              if "bge-m3" in p.lower() and os.path.exists(os.path.join(os.path.dirname(p), "tokenizer_config.json")))
EMB = _emb[0] if _emb else "BAAI/bge-m3"
print("INDEX_DIR:", INDEX_DIR)
print("EMB      :", EMB)

In [ ]:
# Build index (~10-20 phút). Tự bỏ qua nếu đã có meta.json
if os.path.exists(os.path.join(INDEX_DIR, "meta.json")):
    print("Index đã có, bỏ qua build:", INDEX_DIR)
else:
    print("Chưa có index, đang build:", INDEX_DIR)
    !python scripts/build_index.py --embed-model "{EMB}" --index-dir {INDEX_DIR}

In [ ]:
# Smoke test (~5 phút): 20 câu, kiểm tra retrieval + prompt + stop token
!python scripts/run_eval.py --mode rag --n-eval 20 --no-bertscore --exp-name smoke_rag --embed-model "{EMB}" --index-dir {INDEX_DIR} --runs-dir {RUNS_DIR}
!head -n 3 {RUNS_DIR}/smoke_rag/predictions.jsonl

In [ ]:
# E0: closed-book
!python scripts/run_eval.py --mode closed --index-dir {INDEX_DIR} --runs-dir {RUNS_DIR}

In [ ]:
# E1: RAG top-3
!python scripts/run_eval.py --mode rag --top-k 3 --embed-model "{EMB}" --index-dir {INDEX_DIR} --runs-dir {RUNS_DIR}

In [ ]:
# E2: oracle context
!python scripts/run_eval.py --mode oracle --index-dir {INDEX_DIR} --runs-dir {RUNS_DIR}

## E3: RAG + reranker (BAAI/bge-reranker-v2-m3)
Dùng lại **đúng top-10 BGE-M3 của E1** cho từng câu, reranker chấm lại 10 đoạn, lấy top-3 mới cho VinaLLaMA. Prompt, giới hạn token, tham số sinh giữ nguyên như E1.

Nếu chạy ở session mới (không chạy lại E0–E2): *Add Input* một Kaggle Dataset chứa `index/` và `runs/rag_k3_vinallama7b/predictions.jsonl` (ví dụ upload `output.zip`), chạy các cell cài đặt + cell 6 (tìm INDEX_DIR), bỏ qua smoke/E0/E1/E2, rồi chạy cell dưới.

In [ ]:
# E3: RAG + reranker, dùng lại top-10 của baseline E1
_local = f"{RUNS_DIR}/rag_k3_vinallama7b/predictions.jsonl"
_b = sorted(glob.glob("/kaggle/input/**/rag_k3_vinallama7b/predictions.jsonl", recursive=True))
if os.path.exists(_local):
    BASE_PRED = _local                      # E1 vừa chạy trong session này
elif len(_b) == 1:
    BASE_PRED = _b[0]                       # E1 từ Kaggle Dataset đã Add Input
else:
    raise RuntimeError(f"Cần đúng 1 file baseline, tìm thấy {len(_b)}: {_b}")
print("Baseline top-10 lấy từ:", BASE_PRED)
!python scripts/run_eval.py --mode rerank --top-k 3 --candidates-from "{BASE_PRED}" --index-dir {INDEX_DIR} --runs-dir {RUNS_DIR}
!cat {RUNS_DIR}/rag_rerank_top3_vinallama7b/rerank_info.json
!head -n 2 {RUNS_DIR}/rag_rerank_top3_vinallama7b/predictions.jsonl

In [ ]:
!python scripts/summarize.py {RUNS_DIR}
!head -n 3 {RUNS_DIR}/rag_k3_vinallama7b/predictions.jsonl

## Sau khi chạy
- Lưu index để lần sau khỏi encode lại: *Save Version* → Output, rồi tạo Dataset `vimedrag-index` từ thư mục `/kaggle/working/index`.
- Nếu session bị ngắt: chạy lại cùng cell, pipeline tự bỏ qua câu đã xong (resume). Lưu ý `/kaggle/working` mất khi mở session mới nếu chưa Save Version.
- Gửi lại: `runs/*/metrics.json` + vài dòng `predictions.jsonl`.